# Creekside heating DSM — hourly demand surrogate

**sp_creekside · BAS bootstrap → scikit-learn → Excel / later E+ farm**

| | |
|---|---|
| **Question** | Which regressor best predicts `facility_kw` from weather + 6-Area HP occupancy / preheat knobs? |
| **Validation** | `GroupKFold` by **day** (no same-day leakage) |
| **Metrics** | MAE / RMSE overall + **morning peak HE 05–09** |
| **Honesty** | `BAS_BOOTSTRAP_PROXY` · status **CANDIDATE** — not EnergyPlus, not tariff-grade |
| **Ship path** | `ml/artifacts/heating_dsm_hourly_v1.joblib` |

Helpers: `ml/feature_compile_heating_dsm.py`, `ml/train_heating_dsm.py`, `ml/notebook_plots.py`.

## 0 · Setup

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

ROOT = Path("..").resolve()
if not (ROOT / "ml").is_dir():
    ROOT = Path(".").resolve()
ML = ROOT / "ml"
sys.path.insert(0, str(ML))

from artifact_paths import artifact_paths, bootstrap_parquet_path
from feature_compile_heating_dsm import (
    FEATURE_COLS, compile_features, matrix_xy, morning_peak_mask,
    assert_no_future_leakage, cost_from_hourly_kw,
)
from train_heating_dsm import bake_off
from notebook_plots import (
    family_cv_mae_bars, family_mae_rmse_grouped, leaderboard_table,
    oat_vs_kw_scatter, strategy_morning_peak_bars, example_day_profiles,
    residual_hist, save_fig,
)

PATHS = artifact_paths()
PATHS["figures"].mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT)
print("features", len(FEATURE_COLS))

## 1 · Load bootstrap parquet

In [ ]:
pq = bootstrap_parquet_path()
if not pq.is_file():
    import subprocess
    subprocess.check_call([sys.executable, "-u", str(ML / "build_bootstrap_dataset.py")], cwd=str(ROOT))
df = pd.read_parquet(pq)
print(df.shape)
print(df["strategy_id"].value_counts())
df.head(3)

## 2 · Feature compile + leakage guard

In [ ]:
feat = compile_features(df)
assert_no_future_leakage(df)
X, y, groups, cols = matrix_xy(df)
peak = morning_peak_mask(df)
print("X", X.shape, "peak hours", int(peak.sum()), "days", pd.Series(groups).nunique())
feat[FEATURE_COLS].describe().T.head(12)

## 3 · Exploratory figures

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
oat_vs_kw_scatter(df, ax=axes[0])
strategy_morning_peak_bars(df, ax=axes[1])
plt.tight_layout()
save_fig(PATHS["figures"] / "oat_and_strategy_peak.png", fig)
plt.show()

cold = (
    df[df["is_weekend"] < 0.5]
    .groupby("day")["oat_f"].mean()
    .sort_values()
    .index[0]
)
fig, ax = plt.subplots(figsize=(9, 4))
example_day_profiles(df, cold, ax=ax)
save_fig(PATHS["figures"] / "example_cold_day_strategies.png", fig)
plt.show()
print("example day", cold)

## 4 · Model bake-off (GroupKFold)

In [ ]:
result = bake_off(df, n_splits=5)
lb = leaderboard_table(result["leaderboard"], result["cv"]["persistence"])
display(lb)
print("champion:", result["champion"], "| beat persistence:", result["beat_persistence_peak"])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
family_cv_mae_bars(result["leaderboard"], result["cv"]["persistence"]["mae_peak_05_09"], ax=axes[0])
family_mae_rmse_grouped(result["leaderboard"], result["cv"]["persistence"], ax=axes[1])
plt.tight_layout()
save_fig(PATHS["figures"] / "sklearn_leaderboard.png", fig)
plt.show()

## 5 · Residuals + feature importances

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.base import clone

model = result["model"]
oof = np.zeros_like(y)
gkf = GroupKFold(n_splits=result["n_splits"])
for tr, te in gkf.split(X, y, groups):
    m = clone(model)
    m.fit(X[tr], y[tr])
    oof[te] = m.predict(X[te])

fig, ax = plt.subplots(figsize=(6, 4))
residual_hist(y, oof, ax=ax)
save_fig(PATHS["figures"] / "oof_residuals.png", fig)
plt.show()
print("OOF MAE", float(np.mean(np.abs(y - oof))),
      "morning peak MAE", float(np.mean(np.abs(y[peak] - oof[peak]))))

if hasattr(model, "feature_importances_"):
    imp = pd.Series(model.feature_importances_, index=cols).sort_values(ascending=False).head(15)
    fig, ax = plt.subplots(figsize=(7, 5))
    imp.iloc[::-1].plot(kind="barh", ax=ax, color="#2a9d8f")
    ax.set_title("Champion feature importances (top 15)")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    save_fig(PATHS["figures"] / "feature_importances.png", fig)
    plt.show()

## 6 · Midnight 24h forecast demo + cost playground

Replay a cold BAS day as a stand-in for a midnight Open-Meteo forecast. Compare
baseline vs stagger vs 24/7 vs morning-all-on under PLACEHOLDER Madison-ish rates.

In [ ]:
demo_day = cold
rates = {"energy_rate_per_kwh": 0.11, "demand_rate_per_kw": 18.0}

rows = []
for sid in ["baseline", "stagger_preheat", "flat_24_7", "morning_all_on"]:
    sub = df[(df["day"] == demo_day) & (df["strategy_id"] == sid)].sort_values("hour_ending")
    Xd, yd, _, _ = matrix_xy(sub)
    pred = model.predict(Xd)
    cost = cost_from_hourly_kw(pred, **rates)
    cost["strategy_id"] = sid
    cost["true_peak"] = float(sub["facility_kw"].max())
    rows.append(cost)

cost_df = pd.DataFrame(rows).set_index("strategy_id")
display(cost_df)

fig, ax = plt.subplots(figsize=(9, 4))
for sid in cost_df.index:
    sub = df[(df["day"] == demo_day) & (df["strategy_id"] == sid)].sort_values("hour_ending")
    Xd, _, _, _ = matrix_xy(sub)
    ax.plot(sub["hour_ending"], model.predict(Xd), label=sid, lw=1.8)
ax.set_title(f"Model 24h profiles — {demo_day} (PLACEHOLDER rates)")
ax.set_xlabel("Hour local")
ax.set_ylabel("pred facility_kw")
ax.legend(fontsize=8, frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
save_fig(PATHS["figures"] / "forecast_day_cost_profiles.png", fig)
plt.show()

## 7 · Serialize champion

In [ ]:
PATHS["joblib"].parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "model": result["model"],
        "feature_cols": result["feature_cols"],
        "champion": result["champion"],
        "schema": "creekside.heating_dsm_hourly.v1",
    },
    PATHS["joblib"],
)
summary = {
    "champion": result["champion"],
    "beat_persistence_peak": result["beat_persistence_peak"],
    "cv": result["cv"],
    "leaderboard": [
        {"family": e["family"], "oof_metrics": e["oof_metrics"]}
        for e in result["leaderboard"]
    ],
}
PATHS["champion_summary"].write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print("wrote", PATHS["joblib"])
print("wrote", PATHS["champion_summary"])